Import required libraries

In [1]:
import sys
#Append the parent directory to the sys.path to allow imports from neuttsair package
sys.path.append('..')
from neutts import NeuTTS
from IPython.display import Audio

c:\Users\3bdelmoemn\miniconda3\envs\gp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0120 03:23:01.617000 17112 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Downloads files and loads the model into memory

In [13]:
%%capture
tts = NeuTTS(
    backbone_repo="neuphonic/neutts-nano-q8-gguf",
    backbone_device="cpu",
    codec_repo="neuphonic/neucodec",
    codec_device="cpu"
)

Pick your speaker and type up your input text - and generate!

In [14]:
speaker = "jo" # default speakers are 'dave' and 'jo'
input_text = "My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all."

ref_text = f"../samples/{speaker}.txt"
ref_audio_path = f"../samples/{speaker}.wav"

ref_text = open(ref_text, "r").read().strip()
ref_codes = tts.encode_reference(ref_audio_path)
wav = tts.infer(input_text, ref_codes, ref_text)

Listen to your generation!

In [15]:
Audio(wav, rate=24000)

In [ ]:
"""
TTS Controller - Ultra-Fast & Ultra-Smooth Version
أسرع بداية تشغيل + أنعم صوت ممكن
"""

import pyaudio
import numpy as np
from neutts import NeuTTS
import threading
import time
from collections import deque


class TTSController:
    """
    TTS Controller محسّن للسرعة القصوى والـ smoothness
    
    التحسينات:
    - بداية فورية للتشغيل (بدون انتظار chunks)
    - صوت أكثر نعومة ووضوح
    - zero-delay حقيقي
    
    Usage:
    -------
    tts = TTSController()
    tts.setup_voice(audio_path, transcript)
    tts.speak(text)  # فوري وسلس
    """
    
    def __init__(self, model_type="nano"):
        """
        Initialize TTS with maximum speed & smoothness
        
        Parameters:
        -----------
        model_type : str
            "nano" = fastest (recommended)
        """
        
        print(f"🔄 Loading TTS ({model_type})...")
        
        # Model selection
        models = {
            "nano": "neuphonic/neutts-nano",
            "nano-q4": "neuphonic/neutts-nano-q4-gguf",
            "nano-q8": "neuphonic/neutts-nano-q8-gguf",
            "air-q4": "neuphonic/neutts-air-q4-gguf",
            "air-q8": "neuphonic/neutts-air-q8-gguf",
        }
        
        backbone = models.get(model_type, "neuphonic/neutts-nano")
        
        # TTS Model
        self.tts = NeuTTS(
            backbone_repo="neuphonic/neutts-nano-q8-gguf",
            backbone_device="cpu",
            codec_repo="neuphonic/distill-neucodec",
            codec_device="cpu"
        )
        
        # ✨ OPTIMIZED: أفضل إعدادات للسرعة والـ smoothness
        # chunks أصغر = أسرع بداية
        # lookback/lookforward أكبر = صوت أنعم
        self.tts.streaming_frames_per_chunk = 20      # أصغر = أسرع
        self.tts.streaming_lookforward = 8            # زيادة للـ smoothness
        self.tts.streaming_lookback = 50              # زيادة للـ transitions
        self.tts.streaming_stride_samples = self.tts.streaming_frames_per_chunk * self.tts.hop_length
        
        # Voice reference
        self.ref_codes = None
        self.ref_text = None
        self.is_ready = False
        
        # PyAudio setup
        self.audio = pyaudio.PyAudio()
        self.stream = None
        
        # ✨ OPTIMIZED: Buffer أكبر للـ smoothness
        self.audio_buffer = deque(maxlen=100)  # حد أقصى لمنع التراكم
        self.buffer_lock = threading.Lock()
        
        # Control flags
        self.is_speaking = False
        self.is_generating = False
        self.stop_flag = False
        
        print(f"✅ TTS Ready ({model_type})!")
    
    def setup_voice(self, audio_path: str, transcript: str):
        """
        One-time voice encoding
        
        Parameters:
        -----------
        audio_path : str
            Path to voice sample (wav, 3-15 sec recommended)
        transcript : str
            Reference transcript matching the audio
        """
        
        print(f"🎤 Encoding voice...")
        
        self.ref_codes = self.tts.encode_reference(audio_path)
        self.ref_text = transcript
        self.is_ready = True
        
        print("✅ Voice ready!")
    
    def speak(self, text: str, wait=False):
        """
        Ultra-fast, ultra-smooth speech
        
        Parameters:
        -----------
        text : str
            Text from LLM
        wait : bool
            False = non-blocking (default)
            True = blocking until speech completes
        """
        
        if not self.is_ready:
            raise RuntimeError("❌ Call setup_voice() first!")
        
        if not text or not text.strip():
            return
        
        # Stop any previous speech
        self.stop()
        
        # Wait for previous to finish
        while self.is_speaking or self.is_generating:
            time.sleep(0.005)  # أقل delay
        
        # Clear buffer
        with self.buffer_lock:
            self.audio_buffer.clear()
        
        # Reset flags
        self.stop_flag = False
        
        # ✨ KEY OPTIMIZATION: Start playback IMMEDIATELY
        # بدلاً من انتظار chunks، نبدأ التشغيل فوراً
        playback_thread = threading.Thread(
            target=self._play_audio,
            daemon=True
        )
        playback_thread.start()
        
        # Start generation
        gen_thread = threading.Thread(
            target=self._generate_audio,
            args=(text,),
            daemon=True
        )
        gen_thread.start()
        
        if wait:
            # Wait for generation to finish
            while self.is_generating:
                time.sleep(0.01)
            # Wait for playback to finish
            while self.is_speaking:
                time.sleep(0.01)
    
    def _generate_audio(self, text: str):
        """
        Generate audio chunks with zero delay
        Runs in separate thread
        """
        
        self.is_generating = True
        
        try:
            print(f"🔊 Generating: {text[:60]}...")
            
            chunk_count = 0
            
            # Generate chunks and add to buffer immediately
            for chunk in self.tts.infer_stream(
                text,
                self.ref_codes,
                self.ref_text
            ):
                if self.stop_flag:
                    break
                
                # Convert to proper format
                audio_data = chunk.astype(np.float32)
                
                # ✨ INSTANT ADD: أضف للـ buffer فوراً
                with self.buffer_lock:
                    self.audio_buffer.append(audio_data)
                
                chunk_count += 1
            
            print(f"✅ Generated {chunk_count} chunks")
            
        except Exception as e:
            print(f"❌ Generation error: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            self.is_generating = False
    
    def _play_audio(self):
        """
        Play buffered audio with maximum smoothness
        Runs in separate thread
        """
        
        self.is_speaking = True
        
        # ✨ OPTIMIZED: Buffer size أكبر = صوت أنعم
        # 24576 = 1 second buffer للـ smoothness القصوى
        self.stream = self.audio.open(
            format=pyaudio.paFloat32,
            channels=1,
            rate=24000,
            output=True,
            frames_per_buffer=24576  # أكبر buffer = أنعم صوت
        )
        
        try:
            print("🔊 Playing audio...")
            
            empty_buffer_count = 0
            max_empty_waits = 200  # 2 seconds max wait
            
            while True:
                if self.stop_flag:
                    break
                
                # Get chunk from buffer
                with self.buffer_lock:
                    if self.audio_buffer:
                        chunk = self.audio_buffer.popleft()
                        empty_buffer_count = 0  # Reset counter
                    else:
                        # Buffer empty
                        if not self.is_generating:
                            # Generation finished and buffer empty → done
                            break
                        else:
                            # ✨ SMART WAIT: انتظر بذكاء
                            empty_buffer_count += 1
                            
                            # إذا انتظرنا كثير، توقف (safety)
                            if empty_buffer_count > max_empty_waits:
                                print("⚠️ Timeout waiting for audio")
                                break
                            
                            # انتظر chunk جديد
                            time.sleep(0.01)
                            continue
                
                # ✨ SMOOTH PLAYBACK: التشغيل المستمر
                # stream.write هتستنى لو الـ buffer مليان
                # دا بيضمن تشغيل سلس بدون gaps
                self.stream.write(chunk.tobytes())
            
            print("✅ Playback complete")
            
        except Exception as e:
            print(f"❌ Playback error: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            # Close stream properly
            if self.stream:
                try:
                    # ✨ PROPER CLEANUP: تأكد من تشغيل كل الصوت
                    self.stream.stop_stream()
                    self.stream.close()
                except:
                    pass
                self.stream = None
            
            self.is_speaking = False
    
    def stop(self):
        """
        Immediately stop current speech
        """
        self.stop_flag = True
        
        # Clear buffer
        with self.buffer_lock:
            self.audio_buffer.clear()
        
        # Wait for threads to stop
        timeout = time.time() + 1.0
        while (self.is_speaking or self.is_generating) and time.time() < timeout:
            time.sleep(0.01)
    
    def wait_until_done(self):
        """
        Wait until current speech completes
        """
        while self.is_speaking or self.is_generating:
            time.sleep(0.05)
    
    def cleanup(self):
        """
        Clean up resources properly
        """
        
        print("🧹 Cleaning up...")
        
        # Stop any ongoing speech
        self.stop()
        
        # Close stream if still open
        if self.stream:
            try:
                self.stream.stop_stream()
                self.stream.close()
            except:
                pass
        
        # Terminate PyAudio
        if self.audio:
            try:
                self.audio.terminate()
            except:
                pass
        
        # Close TTS model properly
        if hasattr(self.tts, 'backbone'):
            try:
                if hasattr(self.tts.backbone, 'close'):
                    self.tts.backbone.close()
                elif hasattr(self.tts.backbone, '__del__'):
                    del self.tts.backbone
            except:
                pass
        
        print("✅ Cleanup complete!")


# ============================================
# Usage Example - Real Interview Simulation
# ============================================

if __name__ == "__main__":
    
    import time
    
    print("\n" + "="*70)
    print("⚡ ULTRA-FAST & ULTRA-SMOOTH TTS")
    print("="*70 + "\n")
    
    # Initialize
    tts = TTSController(model_type="nano")
    
    # Setup voice
    tts.setup_voice(
        audio_path=r"D:\Education\Voice & Shield\test projects\tts\neutts\samples\jo.wav",
        transcript="So I just tried Neuphonic and I'm genuinely impressed. It's super responsive, it sounds clean, supports voice cloning, and the agent feature is fun to play with too. Highly recommend it for podcasts, conversations, or even just messing around with voiceovers."
    )
    
    print("\n🎤 Interactive Mode - Test the speed!\n")
    print("=" * 70)
    
    # Interactive testing
    while True:
        user_input = input("\nYou: ")
        
        if user_input.lower() in ["exit", "quit", "q"]:
            break
        
        if not user_input.strip():
            continue
        
        # Measure speed
        start_time = time.time()
        print("AI: ", end="", flush=True)
        
        # Speak immediately
        tts.speak(user_input, wait=True)
        
        elapsed = time.time() - start_time
        print(f"\n⏱️  Total time: {elapsed:.2f}s")
        print("-" * 70)
    
    # Cleanup
    tts.cleanup()
    print("\n✅ Done!")

In [3]:
from pprint import pprint
# Interview simulation
interview = [
    {
        "q": "Tell me about yourself.",
        "a": "Good morning! Thank you for this opportunity. I'm excited to discuss my qualifications for the position"
    },
    {
        "q": "What's your experience?",
        "a": "I have over five years of experience in software development, specializing in artificial intelligence and machine learning applications."
    },
    {
        "q": "What are your strengths?",
        "a": "My key strengths include problem solving, team collaboration, and the ability to deliver complex projects on tight deadlines."
    },
    {
        "q": "Why this role?",
        "a": "I'm particularly interested in your artificial intelligence projects and the opportunity to work with cutting-edge technology in a dynamic environment."
    },
    {
        "q": "Are you a good fit?",
        "a": "Absolutely! I believe my technical skills, experience, and passion for innovation make me an excellent fit for this role."
    }
]
    
pprint(interview)

[{'a': "Good morning! Thank you for this opportunity. I'm excited to discuss "
       'my qualifications for the position',
  'q': 'Tell me about yourself.'},
 {'a': 'I have over five years of experience in software development, '
       'specializing in artificial intelligence and machine learning '
       'applications.',
  'q': "What's your experience?"},
 {'a': 'My key strengths include problem solving, team collaboration, and the '
       'ability to deliver complex projects on tight deadlines.',
  'q': 'What are your strengths?'},
 {'a': "I'm particularly interested in your artificial intelligence projects "
       'and the opportunity to work with cutting-edge technology in a dynamic '
       'environment.',
  'q': 'Why this role?'},
 {'a': 'Absolutely! I believe my technical skills, experience, and passion for '
       'innovation make me an excellent fit for this role.',
  'q': 'Are you a good fit?'}]
